# EAST Tokamak — Publication-Quality Manifold Research Notebook

**Workflow**
1. Load EAST EFIT equilibrium + RMP perturbation field
2. Construct the total field `B_tot` (`RegualrCylindricalGridField`)
3. Draw equilibrium overview (Bpol streamplot + ψ contours + first-wall geometry)
4. Trace Poincaré orbits and overlay
5. Locate divertor X-cycles (upper/lower) and island-chain X-cycles via Newton iteration
6. Compute Jacobian evolution along each X-cycle → eigenvalues / eigenvectors
7. Grow stable and unstable manifolds
8. Visualise manifolds on Poincaré cross-section with publication colour convention:
   - **Warm** (Oranges) = unstable manifold $W^{\mathrm{u}}$; hue encodes poloidal arc length $s$
   - **Cool**  (GnBu)   = stable   manifold $W^{\mathrm{s}}$; hue encodes poloidal arc length $s$

---
*Terminology note:*
- **轨道 (trajectory)**: continuous-time path in the 3-D phase space of a field-line flow.
- **踪迹 (orbit)**: discrete sequence of Poincaré section returns — the orbit of the discrete Poincaré map.
- An **X-cycle** is a periodic trajectory of the continuous flow whose Poincaré section returns form a hyperbolic orbit.


## 0. Imports & global settings

In [ ]:
import sys
sys.path.insert(0, r'D:\Repo\pyna')

from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt

from pyna.MCF.visual.tokamak_manifold import (
    PUBLICATION_RC,
    plot_equilibrium_cross_section,
    plot_poincare_orbits,
    plot_xcycle_marker,
    plot_manifold_1d,
    plot_xcycle_all_manifolds,
    make_tokamak_overview_figure,
    manifold_legend_handles,
)

plt.rcParams.update(PUBLICATION_RC)
print('pyna MCF visual OK')

## 1. Load EAST EFIT equilibrium

In [ ]:
# ── Choose shot and time point ───────────────────────────────────────────────
machine  = 'EAST'
shotnum  = 108004
tpoint   = 3.25    # seconds

path_machine = Path(r'D:\Repo\FusionData\EAST')
div_file = path_machine / r'divertor\EAST_divertor.dat'

# ── Equilibrium field on (R, Z, Phi) grid ───────────────────────────────────
import dig.utilvar
R, Z, Phi, BRs, BZs, BPhis = dig.utilvar.get_EFIT_BR_BZ_BPhi(
    machine, shotnum, [tpoint]
)
BR0, BZ0, BPhi0 = BRs[0], BZs[0], BPhis[0]
print(f'R: {R.shape},  Z: {Z.shape},  Phi: {Phi.shape}')
print(f'BR: {BR0.shape}')

## 2. Load RMP perturbation & build total field

In [ ]:
import dig.utilvar, MDSplus

mds_treedict = {}
dig.utilvar.get_tRMP_IRMP(machine, shotnum, mds_treedict)
dig.utilvar.get_limiter(machine, shotnum, mds_treedict)

# ── Coil currents at tpoint ──────────────────────────────────────────────────
from sympy import Float
from mhdpy.field import CoilsysOpMode

ncoil = 8
kAt_expr_dict = {
    **{f'UP{i+1}': Float(
        dig.utilvar.time_linear_weighting(
            mds_treedict['east']['tRMP'],
            mds_treedict['east'][f'IRMPU{i+1}'], tpoint) * 1e-3 * 4)
       for i in range(ncoil)},
    **{f'DOWN{i+1}': Float(
        dig.utilvar.time_linear_weighting(
            mds_treedict['east']['tRMP'],
            mds_treedict['east'][f'IRMPL{i+1}'], tpoint) * 1e-3 * 4)
       for i in range(ncoil)},
}

In [ ]:
# ── Compute perturbation field on the 3-D grid ───────────────────────────────
# (adapt to your mhdpy / dig API as needed)
from mhdpy.field import CoilsysOpMode
import mhdpy.field as mf

BR_pert, BZ_pert, BPhi_pert = mf.compute_RMP_field(
    R, Z, Phi, kAt_expr_dict, machine, shotnum,
    opmode=CoilsysOpMode.VACUUM
)

BR_tot   = BR_pert   + BR0[:, :, None]
BZ_tot   = BZ_pert   + BZ0[:, :, None]
BPhi_tot = BPhi_pert + BPhi0[:, :, None]

In [ ]:
from pyna.MCF.coils.field import RegualrCylindricalGridField
B_tot = RegualrCylindricalGridField(R, Z, Phi, BR_tot, BZ_tot, BPhi_tot)
print(B_tot)

## 3. Equilibrium cross-section overview

In [ ]:
# ── Minimal equilibrium wrapper so plot_equilibrium_cross_section can call BR/BZ ──
import scipy.interpolate as sci

class EastEquilWrapper:
    """Thin wrapper exposing BR / BZ / psi scalar interface from EFIT grids."""
    def __init__(self, R, Z, BR0, BZ0, BPhi0):
        self.R0     = float(R[np.argmin(np.abs(R - R[len(R)//2]))])   # approx
        self.a      = 0.45    # EAST minor radius in m (override if known)
        self.kappa  = 1.7
        self._BR_interp   = sci.RegularGridInterpolator((R, Z), BR0,   bounds_error=False, fill_value=0.0)
        self._BZ_interp   = sci.RegularGridInterpolator((R, Z), BZ0,   bounds_error=False, fill_value=0.0)
        self._BPhi_interp = sci.RegularGridInterpolator((R, Z), BPhi0, bounds_error=False, fill_value=1.0)

    def BR(self, r, z):
        return float(self._BR_interp([[r, z]]))

    def BZ(self, r, z):
        return float(self._BZ_interp([[r, z]]))

    def Bphi(self, r):
        return float(self._BPhi_interp([[r, 0.0]]))


eq_wrapper = EastEquilWrapper(R, Z, BR0, BZ0, BPhi0)

# ── Read divertor geometry ───────────────────────────────────────────────────
import mhdpy.file
div_edge = mhdpy.file.read_divertor(div_file)   # returns (N,2) or list of arrays

fig_ov, ax_ov = plot_equilibrium_cross_section(
    eq_wrapper,
    n_surfaces=22,
    n_grid=280,
    streamplot_density=2.0,
    divertor_RZ=div_edge if isinstance(div_edge, np.ndarray) else None,
    show_colorbar=True,
)
ax_ov.set_title(f'EAST #{shotnum} @ {tpoint} s', fontsize=10)
plt.tight_layout()
plt.show()

## 4. Poincaré orbits (dense tracing, φ = 0 section)

In [ ]:
import pyna.flt

# ── Dense initial conditions along midplane ───────────────────────────────────
n_fl = 60
initpts_RZPhi = np.column_stack([
    np.linspace(1.30, 2.10, n_fl),
    np.zeros(n_fl),
    np.zeros(n_fl),
])

# --- Cache path ---------------------------------------------------------------
cache_file = path_machine / 'equili' / f'EAST_{shotnum}_{int(tpoint*1000)}ms_Poincare.npz'

if cache_file.exists():
    Poincare_trace_list = pyna.flt.load_Poincare_orbits(str(cache_file))
    print(f'Loaded {len(Poincare_trace_list)} traces from cache.')
else:
    n_turns   = 200
    flt_result = pyna.flt.bundle_tracing_with_t_as_DeltaPhi(
        B_tot, 2 * np.pi * n_turns, initpts_RZPhi,
        phi_increasing=True,
        max_step=(Phi[1] - Phi[0]) / 2,
    )
    from pyna.topo.manifold import _transect_initPhi0_Wivp_at_a_phi
    all_crossings = _transect_initPhi0_Wivp_at_a_phi(flt_result, phi=0.0, mturn=1)
    Poincare_trace_list = [
        all_crossings[i::n_fl, :] for i in range(n_fl)
    ]
    cache_file.parent.mkdir(exist_ok=True)
    pyna.flt.save_Poincare_orbits(str(cache_file), Poincare_trace_list)
    print(f'Traced & saved {len(Poincare_trace_list)} field lines.')

In [ ]:
fig_pc, ax_pc = plot_equilibrium_cross_section(
    eq_wrapper,
    n_grid=220,
    streamplot_density=1.5,
    show_colorbar=False,
    n_surfaces=18,
)

# ψ_norm for each field line (used for colour)
psi_norm_starts = np.linspace(0.0, 1.0, n_fl)  # refine if you have psi on grid

sm = plot_poincare_orbits(
    Poincare_trace_list, ax_pc,
    psi_values=psi_norm_starts,
    cmap='viridis', s=0.25, alpha=0.70,
)

from mpl_toolkits.axes_grid1 import make_axes_locatable
divider = make_axes_locatable(ax_pc)
cax     = divider.append_axes('right', size='4%', pad=0.05)
cb      = fig_pc.colorbar(sm, cax=cax)
cb.set_label(r'$\psi_{\rm norm}$', fontsize=10)

ax_pc.set_title(f'Poincaré orbits — EAST #{shotnum} @ {tpoint} s  (φ = 0)', fontsize=10)
plt.tight_layout()
plt.show()

## 5. Locate X-cycles via Newton iteration

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import pyna.diff.fixedpoint as fpmod

def EAST_SOL_inside(R_, Z_):
    return 1.25 < R_ < 2.55 and -1.00 < Z_ < 1.152

tasks = {
    'UP_X'   : ((B_tot, [1.6,  0.75, 0.0]), dict(tor_turn=1, zone_of_interest_callback=EAST_SOL_inside, h=1.0, epsilon=1e-6)),
    'DOWN_X' : ((B_tot, [1.6, -0.75, 0.0]), dict(tor_turn=1, zone_of_interest_callback=EAST_SOL_inside, h=1.0, epsilon=1e-6)),
    'AXIS_O' : ((B_tot, [1.86, 0.0,  0.0]), dict(tor_turn=1, zone_of_interest_callback=EAST_SOL_inside, h=1.0, epsilon=1e-6)),
    # Add m=3 island chain X/O if relevant:
    # 'm3_X' : ((B_tot, [1.83, -0.17, 0.0]), dict(tor_turn=3, zone_of_interest_callback=EAST_SOL_inside, h=1.0, epsilon=1e-6)),
}

fixed_points = {}
with ThreadPoolExecutor() as pool:
    futures = {
        pool.submit(fpmod.Newton_discrete, *args, **kwargs): name
        for name, (args, kwargs) in tasks.items()
    }
    for fut in as_completed(futures):
        name = futures[fut]
        result = fut.result()
        fixed_points[name] = result
        print(f'{name}: R={result[0]:.4f} m, Z={result[1]:.4f} m')

In [ ]:
# Overlay fixed points on Poincaré figure
for name, RZ in fixed_points.items():
    stability = 'hyperbolic' if 'X' in name else 'elliptic'
    plot_xcycle_marker(
        np.array([RZ[0], RZ[1]]),
        stability=stability,
        ax=ax_pc,
        label=name.replace('_', ' '),
    )

ax_pc.legend(fontsize=8, framealpha=0.85, markerscale=1.2)
fig_pc

## 6. Jacobian evolution along X-cycles

In [ ]:
from pyna.diff.fieldline import RZ_partial_derivative_of_map_4_Flow_Phi_as_t
from pyna.diff.cycle import Jac_evolution_along_cycle, eigvec_interpolator_along_Xcycle
from pyna.diff.fixedpoint import draw_eig_direction

# ── Down-divertor X-cycle (m_turn = 1) ───────────────────────────────────────
mturn_div = 1
DOWNX_RZdiff = RZ_partial_derivative_of_map_4_Flow_Phi_as_t(
    B_tot,
    [0.0, 2 * mturn_div * np.pi],
    fixed_points['DOWN_X'],
    highest_order=1,
    t_eval=[2 * mturn_div * np.pi],
)
DOWNX_DPm_sol = Jac_evolution_along_cycle(
    B_tot, DOWNX_RZdiff,
    Phi_span=[0.0, 2 * mturn_div * np.pi],
)

eigval_interp, eigvec_interp = eigvec_interpolator_along_Xcycle(DOWNX_DPm_sol)
print('DOWN X eigenvalues at phi=0:', eigval_interp(0.0))
# Eigenvalue > 1 → unstable direction; 0 < λ < 1 → stable direction

In [ ]:
# Draw eigenvector arrows at phi=0 on the overview
draw_eig_direction(
    DOWNX_RZdiff[0].sol(0.0),
    np.arctan2(eigvec_interp(0.0)[1, 0], eigvec_interp(0.0)[0, 0]),
    eigval_interp(0.0)[0],
    fig_pc, ax_pc, unit_data_len=0.03, circ_wanted=True,
)
draw_eig_direction(
    DOWNX_RZdiff[0].sol(0.0),
    np.arctan2(eigvec_interp(0.0)[1, 1], eigvec_interp(0.0)[0, 1]),
    eigval_interp(0.0)[1],
    fig_pc, ax_pc, unit_data_len=0.03, circ_wanted=False,
)
fig_pc

## 7. Grow stable / unstable manifolds

In [ ]:
from pyna.topo.manifold import grow_manifold_from_Xcycle_naive_init_segment
from multiprocessing import Pool

# Manifold growth parameters — adjust for desired arm length
total_dPhi = 16 * 2 * np.pi   # total toroidal angle to trace
max_step   = (Phi[1] - Phi[0]) / 2

# eigind=1 → unstable arm +,  eigind=3 → unstable arm -
# eigind=2 → stable   arm +,  eigind=4 → stable   arm -
# (convention from pyna.topo.manifold)

manifold_params = [
    # (eigind, label)
    (1, 'u+'),   # unstable +
    (3, 'u-'),   # unstable -
    (2, 's+'),   # stable   +
    (4, 's-'),   # stable   -
]

DOWNX_W_bundles = {}
with Pool(processes=4) as pool:
    results = {}
    for eigind, label in manifold_params:
        results[label] = pool.apply_async(
            grow_manifold_from_Xcycle_naive_init_segment,
            args=(B_tot, DOWNX_RZdiff, DOWNX_DPm_sol),
            kwds=dict(
                eigind=eigind,
                Phi_span=[0.0, 2 * mturn_div * np.pi],
                total_deltaPhi=total_dPhi,
                ptsnum_initseg=300,
                initseg_len=1.0e-4,
                max_step=max_step,
            ),
        )
    for label, res in results.items():
        DOWNX_W_bundles[label] = res.get()
        print(f'  {label} done')

print('All manifold arms computed.')

## 8. Publication figure — manifolds on Poincaré cross-section

In [ ]:
# ── Two-panel overview + divertor zoom ───────────────────────────────────────
fig_pub, axes = make_tokamak_overview_figure(
    eq_wrapper,
    figsize=(11, 7),
    suptitle=f'EAST #{shotnum} @ {tpoint} s — RMP vacuum manifolds',
)
ax_ov2 = axes['overview']
ax_det = axes['detail']

# --- Left panel: equilibrium + Poincaré orbits --------------------------------
plot_equilibrium_cross_section(
    eq_wrapper, ax=ax_ov2,
    n_grid=220, streamplot_density=1.6,
    show_colorbar=True, n_surfaces=18,
)
plot_poincare_orbits(
    Poincare_trace_list, ax_ov2,
    psi_values=psi_norm_starts,
    cmap='viridis', s=0.3, alpha=0.65,
)
plot_xcycle_marker(
    np.array([[fixed_points['DOWN_X'][0], fixed_points['DOWN_X'][1]],
              [fixed_points['UP_X'][0],   fixed_points['UP_X'][1]]]),
    stability='hyperbolic', ax=ax_ov2, label='X-point',
)
ax_ov2.set_title('Equilibrium overview', fontsize=9)

# --- Right panel: manifolds in divertor region --------------------------------
# Set zoom to lower divertor
R_Xdown, Z_Xdown = fixed_points['DOWN_X']
pad_R, pad_Z = 0.12, 0.10
ax_det.set_xlim(R_Xdown - pad_R, R_Xdown + pad_R)
ax_det.set_ylim(Z_Xdown - pad_Z, Z_Xdown + pad_Z)

# Background: light equilibrium
plot_equilibrium_cross_section(
    eq_wrapper, ax=ax_det,
    n_grid=180, streamplot_density=1.0,
    show_colorbar=False, n_surfaces=12,
    show_lcfs=False,
    xlim=(R_Xdown - pad_R, R_Xdown + pad_R),
    ylim=(Z_Xdown - pad_Z, Z_Xdown + pad_Z),
)

# Manifold arms
mf_lcs = plot_xcycle_all_manifolds(
    fig_pub, ax_det, DOWNX_W_bundles,
    phi=0.0, mturn=mturn_div,
    unstable_cmap='Oranges',
    stable_cmap='GnBu',
    lw=1.1, alpha=0.92,
    show_colorbar=True,
)

# X-point marker
plot_xcycle_marker(
    np.array([R_Xdown, Z_Xdown]),
    stability='hyperbolic', ax=ax_det,
    label='Lower X-point',
)

ax_det.set_aspect('equal')
ax_det.set_title('Lower divertor — manifolds (φ = 0)', fontsize=9)
ax_det.set_xlabel(r'$R\ (\mathrm{m})$', fontsize=10)
ax_det.set_ylabel(r'$Z\ (\mathrm{m})$', fontsize=10)

# Shared legend
handles = manifold_legend_handles()
fig_pub.legend(
    handles=handles,
    loc='lower center', ncol=2, fontsize=8,
    framealpha=0.9, bbox_to_anchor=(0.5, -0.06),
)

plt.tight_layout()
plt.show()

In [ ]:
# ── Save publication figure ───────────────────────────────────────────────────
out_png = path_machine / f'figures/EAST_{shotnum}_{int(tpoint*1000)}ms_manifolds.png'
out_png.parent.mkdir(exist_ok=True)
fig_pub.savefig(out_png, dpi=300, bbox_inches='tight')
print(f'Saved: {out_png}')

## 9. Zoom and multi-arm comparison (optional)

In [ ]:
# ── Single-arm detail: unstable manifold only, reveal folding structure ────────
from pyna.topo.manifold import _transect_initPhi0_Wivp_at_a_phi
from pyna.MCF.visual.tokamak_manifold import plot_manifold_1d

fig_zoom, ax_zoom = plt.subplots(figsize=(6, 6), dpi=150)
ax_zoom.set_aspect('equal')
ax_zoom.set_xlabel(r'$R$ (m)', fontsize=10)
ax_zoom.set_ylabel(r'$Z$ (m)', fontsize=10)
ax_zoom.set_title('Unstable manifold arms — lower X-point (φ = 0)', fontsize=9)

for label in ('u+', 'u-'):
    if label not in DOWNX_W_bundles:
        continue
    W1d = _transect_initPhi0_Wivp_at_a_phi(
        DOWNX_W_bundles[label], phi=0.0, mturn=mturn_div
    )
    plot_manifold_1d(
        fig_zoom, ax_zoom, W1d[:, :2],
        unstable=True, cmap='Oranges',
        lw=1.2, alpha=0.93,
        s_norm_gamma=0.4,
        show_colorbar=(label == 'u+'),
        colorbar_label=r'$s_{\rm unstable}$ (m)',
    )

for label in ('s+', 's-'):
    if label not in DOWNX_W_bundles:
        continue
    W1d = _transect_initPhi0_Wivp_at_a_phi(
        DOWNX_W_bundles[label], phi=0.0, mturn=mturn_div
    )
    plot_manifold_1d(
        fig_zoom, ax_zoom, W1d[:, :2],
        unstable=False, cmap='GnBu',
        lw=1.2, alpha=0.88,
        s_norm_gamma=0.4,
    )

plot_xcycle_marker(
    np.array([R_Xdown, Z_Xdown]),
    stability='hyperbolic', ax=ax_zoom,
)

# auto-zoom to the manifold extent
ax_zoom.autoscale_view()
plt.tight_layout()
plt.show()

## 10. Appendix — island-chain X-cycle (optional)

In [ ]:
# Uncomment and adapt if you have an m=3 (or other) island chain

# mturn_isl = 3
# m3_X_RZ = fixed_points['m3_X']   # requires 'm3_X' in tasks above
#
# m3_X_RZdiff = RZ_partial_derivative_of_map_4_Flow_Phi_as_t(
#     B_tot, [0.0, 2*mturn_isl*np.pi], m3_X_RZ,
#     highest_order=1, t_eval=[2*mturn_isl*np.pi]
# )
# m3_X_DPm_sol = Jac_evolution_along_cycle(
#     B_tot, m3_X_RZdiff, Phi_span=[0.0, 2*mturn_isl*np.pi]
# )
#
# # ... grow manifolds exactly as Section 7 above, then plot with
# # plot_xcycle_all_manifolds with different cmaps if desired.